# 05 ML

Puntúa las anomalías de velocidad de los días de oro que cambiaron. El job ejecuta este notebook después de `04_gold`. No tiene otros modos.

Hacen falta al menos 100 buques-día en los 14 días anteriores al día puntuado. El primer día en oro queda como `sin_historial` y no crea experimento ni modelo. El experimento, cuando existe, es `/Shared/OceanWatch_eq8_velocidad`. El modelo es `oceanwatch_detector_velocidad` en `default`.


In [ ]:
# Capturar los parámetros del notebook llamador antes de %run.
dbutils.widgets.text("fecha", "2023-06-08")
dbutils.widgets.text("ancla_replay", "")
dbutils.widgets.text("inicio_run", "")
ow_argumentos = tuple(dbutils.widgets.get(k).strip() for k in ("fecha", "ancla_replay", "inicio_run"))


In [ ]:
%run ./_config


## Detección de anomalías de velocidad
La unidad de evaluación es un buque en un día UTC. La SOG media procede del AIS. La segunda velocidad divide la distancia acumulada entre posiciones por el tiempo transcurrido entre primera y última señal, incluyendo pausas. Su diferencia ayuda a detectar saltos de posición o velocidades inusuales aun cuando la SOG de cada mensaje esté dentro del rango permitido. Un aviso requiere revisión.

Se usan las dos velocidades y la brecha absoluta, con transformación logarítmica para la velocidad por tramos y la brecha. Isolation Forest aprende con los 14 días anteriores disponibles. El corte es estrictamente anterior al día puntuado. La muestra de entrenamiento tiene un límite de 30.000 trayectorias, un mínimo de 100 y semilla 42. El contraste es el percentil 99 de SOG del entrenamiento. Se registran cobertura, avisos y coincidencias. La fuente carece de etiquetas de anomalías confirmadas.

La versión se registra con firma y ejemplo de entrada. En inferencia se vuelve a cargar por versión. El alias `champion` identifica la versión vigente. Los días con menos de diez minutos entre señales quedan como `traza_insuficiente`; un arranque sin historia queda como `sin_historial`.


In [ ]:
import hashlib
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import IsolationForest

VARIABLES = ["sog_media", "log_velocidad_tramos", "log_brecha"]
PARAM = dict(n_estimators=120, contamination=0.015, max_samples="auto", random_state=42, n_jobs=1)
MIN_ENTRENO, MAX_ENTRENO = 100, 30000
VERSION_FEATURES = "velocidad-v1"


def matriz(df):
    """La escala logarítmica conserva extremos cinemáticos sin desbordar las distancias."""
    return pd.DataFrame({"sog_media": df.sog_media.astype("float64"),
                         "log_velocidad_tramos": np.log1p(df.velocidad_tramos.astype("float64")),
                         "log_brecha": np.log1p(df.brecha.astype("float64"))})[VARIABLES]


def huella_muestra(pdf):
    import json
    orden = pdf.sort_values(["fecha", "MMSI"])[["fecha", "MMSI", "sog_media", "velocidad_tramos", "brecha"]]
    # El redondeo también se aplica a los atributos del modelo, para que una
    # distinta asociación de sumas en Spark no cree versiones por ruido numérico.
    binario = orden.to_csv(index=False, float_format="%.6f", lineterminator="\n").encode()
    receta = json.dumps([VERSION_FEATURES, PARAM, sklearn.__version__], sort_keys=True).encode()
    return hashlib.sha256(receta + binario).hexdigest()


def entrenar(pdf):
    modelo = IsolationForest(**PARAM).fit(matriz(pdf))
    return modelo, float(pdf["sog_media"].quantile(0.99))


def puntuaciones(modelo, pdf):
    return -modelo.decision_function(matriz(pdf))


In [ ]:
def atributos(sesion, tabla):
    from pyspark.sql import functions as F
    t = sesion.table(tabla)
    horas = (F.col("fin").cast("long") - F.col("inicio").cast("long")) / 3600.0
    util = (F.col("n_posiciones") >= 2) & (horas >= 1 / 6) & F.col("sog_media").isNotNull()
    return (t.withColumn("velocidad_tramos", F.when(util, F.round(F.col("distancia_km") / (1.852 * horas), 6)))
            .withColumn("sog_media", F.round("sog_media", 6))
            .withColumn("brecha", F.round(F.abs(F.col("velocidad_tramos") - F.col("sog_media")), 6))
            .select("fecha", "MMSI", "n_posiciones", "sog_media", "velocidad_tramos", "brecha"))


def version_para_muestra(cliente, experimento, nombre, muestra, dia):
    import mlflow
    from mlflow.models import infer_signature
    from mlflow.exceptions import MlflowException
    firma = huella_muestra(muestra)
    try:
        versiones = cliente.search_model_versions(f"name = '{nombre}'", max_results=1000)
    except MlflowException as exc:
        if exc.error_code not in {"RESOURCE_DOES_NOT_EXIST", "NOT_FOUND"}:
            raise
        versiones = []
    for mv in versiones:
        if mv.run_id:
            prev = cliente.get_run(mv.run_id)
            if prev.data.tags.get("huella") == firma:
                return str(mv.version), mv.run_id, float(prev.data.params["umbral_base_p99"])
    modelo, q99 = entrenar(muestra)
    ej = matriz(muestra.iloc[:5])
    with mlflow.start_run(experiment_id=experimento.experiment_id, run_name="Entrenamiento " + str(dia)) as run:
        mlflow.set_tags({"huella": firma, "version_atributos": VERSION_FEATURES, "fuente": GOLD_TRAY,
                         "unidad": "buque-dia", "fecha_corte": str(dia)})
        mlflow.log_params({**PARAM, "inicio_train": str(min(muestra.fecha)), "fin_train": str(max(muestra.fecha)),
                           "umbral_base_p99": q99, "variables": ",".join(VARIABLES),
                           "sklearn": sklearn.__version__, "ventana_dias": 14})
        mlflow.log_metric("filas_entrenamiento", len(muestra))
        dataset = mlflow.data.from_pandas(muestra[["fecha","MMSI","sog_media","velocidad_tramos","brecha"]],
                                          name="trayectorias_oro", digest=firma[:16])
        mlflow.log_input(dataset, context="training")
        mlflow.log_dict({"variables": VARIABLES, "llave": ["fecha", "MMSI"], "umbral_alerta": 0.0,
                         "huella": firma}, "contrato_entrada.json")
        datos = {"name": "detector"} if int(mlflow.__version__.split(".")[0]) >= 3 else {"artifact_path": "detector"}
        info = mlflow.sklearn.log_model(modelo, signature=infer_signature(ej, modelo.predict(ej)), input_example=ej, **datos)
        mv = mlflow.register_model(info.model_uri, nombre)
        for principal in ("`mine-group8`", "`h.benitez@uniandes.edu.co`", "`l.galvan@uniandes.edu.co`"):
            spark.sql(f"GRANT EXECUTE ON MODEL {nombre} TO {principal}")
        return str(mv.version), run.info.run_id, q99


def puntuar_dia(fecha_dia, registro, ruta_experimento):
    fecha = fecha_dia.isoformat()
    import datetime as dt
    import mlflow
    from mlflow import MlflowClient
    from mlflow.exceptions import MlflowException
    from pyspark.sql import functions as F

    inicio = fecha_dia - dt.timedelta(days=14)
    base = atributos(spark, GOLD_TRAY)
    jornada = base.where(F.col("fecha") == F.lit(fecha_dia))
    if jornada.limit(1).count() == 0:
        raise ValueError(f"Falta la trayectoria oro de {fecha}. Ejecuta 04_gold para la fecha.")
    entreno = (base.where((F.col("fecha") >= F.lit(inicio)) & (F.col("fecha") < F.lit(fecha_dia))
                          & F.col("brecha").isNotNull())
               .orderBy(F.xxhash64("fecha", "MMSI"), "fecha", "MMSI")
               .limit(MAX_ENTRENO).toPandas())
    columnas = ["fecha","MMSI","n_posiciones","sog_media","velocidad_tramos","brecha",
                "score","es_anomalia","estado","modelo_version","run_id"]
    entreno_ok = False
    if len(entreno) < MIN_ENTRENO:
        salida = (jornada.withColumn("score", F.lit(None).cast("double"))
                  .withColumn("es_anomalia", F.lit(None).cast("boolean"))
                  .withColumn("estado", F.when(F.col("brecha").isNull(), "traza_insuficiente").otherwise("sin_historial"))
                  .withColumn("modelo_version", F.lit(None).cast("string"))
                  .withColumn("run_id", F.lit(None).cast("string")))
        print(f"Arranque histórico de {fecha}: {len(entreno)} ejemplos anteriores; se requieren {MIN_ENTRENO}.")
        entreno_ok = False
    else:
        mlflow.set_registry_uri(registro)
        exp = mlflow.set_experiment(ruta_experimento or "/Shared/OceanWatch_eq8_velocidad")
        cliente = MlflowClient()
        version, run_id, q99 = version_para_muestra(cliente, exp, MODELO, entreno, fecha_dia)
        # Un backfill usa su versión por fecha sin desplazar una versión más reciente.
        otro_corte = ""
        if "champion" in (cliente.get_registered_model(MODELO).aliases or {}):
            anterior = cliente.get_model_version_by_alias(MODELO, "champion")
            otro_corte = cliente.get_run(anterior.run_id).data.tags.get("fecha_corte", "")
        if fecha >= otro_corte:
            cliente.set_registered_model_alias(MODELO, "champion", version)
        modelo = mlflow.sklearn.load_model(f"models:/{MODELO}/{version}")
        @F.pandas_udf("double")
        def puntuar(sog: pd.Series, velocidad: pd.Series, brechas: pd.Series) -> pd.Series:
            bloque = pd.DataFrame({"sog_media":sog,"velocidad_tramos":velocidad,"brecha":brechas})
            elegibles = bloque.notna().all(axis=1)
            scores = pd.Series(np.nan, index=bloque.index, dtype="float64")
            if elegibles.any():
                scores.loc[elegibles] = puntuaciones(modelo, bloque.loc[elegibles])
            return scores
        salida = (jornada.withColumn("score", puntuar("sog_media", "velocidad_tramos", "brecha"))
                  .withColumn("es_anomalia", F.when(F.col("brecha").isNotNull(), F.col("score") > 0))
                  .withColumn("estado", F.when(F.col("brecha").isNull(), "traza_insuficiente").otherwise("evaluado"))
                  .withColumn("modelo_version", F.lit(version)).withColumn("run_id", F.lit(run_id)))
    (salida.select(*columnas).write.format("delta").mode("overwrite")
     .option("replaceWhere", f"fecha = '{fecha}'").saveAsTable(GOLD_ANOM))
    if len(entreno) >= MIN_ENTRENO:
        reales = spark.table(GOLD_ANOM).where(F.col("fecha") == F.lit(fecha_dia))
        met = reales.agg(F.count("*").alias("total"),F.count("es_anomalia").alias("evaluados"),
            F.count(F.when(F.col("es_anomalia"),1)).alias("alertas_modelo"),
            F.count(F.when((F.col("estado")=="evaluado") & (F.col("sog_media")>q99),1)).alias("alertas_base"),
            F.count(F.when(F.col("es_anomalia") & (F.col("sog_media")>q99),1)).alias("coinciden")).first()
        with mlflow.start_run(experiment_id=exp.experiment_id, run_name="Evaluación temporal " + fecha):
            mlflow.set_tags({"fecha_evaluacion":fecha,"run_entrenamiento":run_id,"modelo_version":version})
            mlflow.log_metrics({**met.asDict(),"cobertura":met.evaluados/met.total,
                                "tasa_alertas":met.alertas_modelo/max(1,met.evaluados)})
        print("Modelo", MODELO, "versión", version, "run", run_id, met.asDict())
        entreno_ok = True
    return entreno_ok


def ejecutar_ml(registro="databricks-uc", ruta_experimento=None):
    # Si entra un mensaje tardío, cambian tanto su día como las ventanas posteriores que lo usaron.
    import json
    receta = hashlib.sha256(json.dumps([VERSION_FEATURES, PARAM, sklearn.__version__, MIN_ENTRENO, MAX_ENTRENO],sort_keys=True).encode()).hexdigest()
    pendientes = spark.sql(f"""WITH firmas AS (
       SELECT t.fecha, sha2(concat('{receta}',concat_ws('|',sort_array(collect_list(
          concat(cast(h.fecha as string),'=',cast(h.batch_plata as string)))))),256) revision
       FROM {CONTROL_ORO} t JOIN {CONTROL_ORO} h
       ON h.fecha BETWEEN date_sub(t.fecha,14) AND t.fecha GROUP BY t.fecha)
       SELECT f.fecha,f.revision FROM firmas f LEFT JOIN {CONTROL_ML} m ON f.fecha=m.fecha
       WHERE m.fecha IS NULL OR f.revision<>m.revision_oro ORDER BY f.fecha""").collect()
    hubo_modelo = False
    for p in pendientes:
        hubo_modelo = puntuar_dia(p.fecha, registro, ruta_experimento) or hubo_modelo
        spark.sql(f"""MERGE INTO {CONTROL_ML} d USING (SELECT DATE '{p.fecha.isoformat()}' fecha,
             '{p.revision}' revision_oro,current_timestamp() publicado_en) s ON d.fecha=s.fecha
             WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *""")
    if not spark.table(CONTROL_ML).where(f"fecha = DATE '{fecha}'").take(1):
        raise ValueError(f"El día {fecha} aún no tiene cierre de oro. Ejecuta 04_gold.")
    print(f"Fechas de ML actualizadas: {len(pendientes)}. Día del job: {fecha}.")
    if pendientes and not hubo_modelo:
        print(
            "No se creó experimento ni modelo. El día puntuado no tenía 100 buques-día "
            "en los 14 días anteriores. Con un solo día en oro el estado es sin_historial. "
            "Ejecuta 04 y este notebook para un segundo día."
        )


In [ ]:
ejecutar_ml()
